In [ ]:
#| default_exp core

# core

In [ ]:
#| export
from collections import defaultdict

In [ ]:
#| export
import numpy as np

In [ ]:
#| export
from fastcore.basics import *

In [ ]:
#| export
import pandas as pd

In [ ]:
#| hide
from portfolio.sample_data import *

In [ ]:
#| hide
from fastcore.test import test_fail

In [ ]:
#| export
def _slice_dates(data, start, end):
    if not start: start = data.index.min()
    if not end: end = data.index.max()
    return data.loc[start:end]

In [ ]:
#| export
def _validate(name, df):
    if df is not None:
        is_monthly = isinstance(df.index, pd.PeriodIndex) and df.index.freqstr == 'M'
        if not is_monthly: raise ValueError(f'{name} df has an index that is not a period index with monthly frequency. Resample your data to monthly frequency')

## A portfolio

To create a portfolio you simply pass the return streams and weights to the portfolio class. It expects monthly return streams. The best way is to gather all your raw, monthly return streams in a single dataframe, including the risk-free (cash-rate) asset. That way you can check for gaps in the data and validate that the data lines up properly before passing it to the portfolio.

The risk free rate will be used to transform the return streams to excess return streams (in excess of risk free).

In [ ]:
#| export
class Portfolio:
    def __init__(self, name:str, return_streams:pd.DataFrame, weights:dict, rf:pd.DataFrame=None, cpi:pd.DataFrame=None, funding_spread:float=0.0, start=None, end=None):
        for n,df in zip(('return_streams', 'rf', 'cpi'),(return_streams, rf, cpi)): _validate(n,df)

        self.name = name
        self.weights = weights
        self.funding_spread = funding_spread
        self.full_rf = rf.copy() if rf is not None else None
        self.full_cpi = cpi.copy() if cpi is not None else None
        self.full_rets = return_streams[list(weights.keys())].copy()

        self.return_streams = _slice_dates(self.full_rets, start, end)
        self.rf = self.full_rf.reindex(self.return_streams.index) if self.full_rf is not None else None
        self.cpi = self.full_cpi.reindex(self.return_streams.index) if self.full_cpi is not None else None
    def __repr__(self): return str(self.weights)

Let's generate sample data for use in analysis

In [ ]:
rets, rf, cpi = sample_data_se()

In [ ]:
rets.head(2)

In [ ]:
rf.head(2)

In [ ]:
cpi.head(2)

We make a simple 60/40 portfolio

In [ ]:
p = Portfolio('60/40', rets, {'bonds': .40, 'stocks': .60}, rf=rf, cpi=cpi)
p

In [ ]:
p.name

In [ ]:
#| export
def _availability_row(df):
    valid = df.dropna().index
    expected = pd.period_range(valid.min(), valid.max(), freq="M")
    return {
        "start": valid.min(),
        "end": valid.max(),
        "observations": len(valid),
        "missing_months": len(expected.difference(valid)),
    }

Sometimes we want to inspect the start and end dates of the raw data that a portfolio uses and whether there are missing data points.

In [ ]:
#| export
@patch(as_prop=True)
def availability(self: Portfolio):
    "Information on the underlying data powering the analysis"
    rows = {"returns": _availability_row(self.full_rets)}
    if self.full_rf is not None: rows["rf"] = _availability_row(self.full_rf)
    if self.full_cpi is not None:rows["cpi"] = _availability_row(self.full_cpi)
    return pd.DataFrame.from_dict(rows, orient="index")

In [ ]:
p.availability

It is also convenient to be able to examine performance in different time periods.

In [ ]:
#| export
@patch
def between(self:Portfolio, start=None, end=None):
    "Changes performance time period from start to end"
    return Portfolio(self.name, self.full_rets, self.weights, rf=self.full_rf, cpi=self.full_cpi, funding_spread=self.funding_spread, start=start, end=end)


In [ ]:
p.return_streams.index.min(), p.return_streams.index.max()

If we want to limit our analysis to between 2010 and 2020 we do like this

In [ ]:
p = p.between('1/1995', '1/2000')

In [ ]:
p.return_streams.index.min(), p.return_streams.index.max()

It is possible to re-slice again to a larger window

In [ ]:
p = p.between('1/1900', '1/2024')

In [ ]:
p.return_streams.index.min(), p.return_streams.index.max()

The portfolio return is the excess return your portfolio makes each month

In [ ]:
#| export
@patch()
def _excess_return(self:Portfolio):
    if self.rf is None: raise ValueError("Cannot calculate excess returns: no risk-free rate was supplied.")
    return self.return_streams.sub(self.rf, axis=0)

In [ ]:
#| export
@patch()
def port_rets(self:Portfolio, excess=True, agg=True):
    'Monthly portfolio return'
    stream = self._excess_return() if excess else self.return_streams
    r = (stream*self.weights)
    if agg: r = r.sum(axis=1, min_count=len(self.weights))
    r.name = self.name
    return r

In [ ]:
p.port_rets().head(2)

If we do not supply a rf, the excess return does not work

In [ ]:
test_fail(
    lambda: Portfolio('60/40', rets, {'bonds': .40, 'stocks': .60}).port_rets(),
    contains="no risk-free rate"
)

## Portfolio Metrics and Stats

In [ ]:
#| export
def _sharpe(rets: pd.DataFrame):
    'Calculates annualised excess return, volatility and sharpe'
    exp_r = rets.mean() * 12
    vol = rets.std() * 12**0.5
    sharpe = exp_r/vol
    return exp_r,vol,sharpe

In [ ]:
#| export
@patch
def stats(self:Portfolio):
    'Excess return, volatility and sharpe ratio (annualised)'
    e,v,s = _sharpe(self.port_rets())
    stats = {'expected_return': e, 'volatility': v, 'sharpe_ratio': s}
    return pd.Series(stats, name=self.name)

In [ ]:
s = p.stats()
s

In [ ]:
#| export
@patch
def cum_return(self:Portfolio):
    'Cumulative compounded total return'
    r = self.port_rets(excess=False)
    return (1+r).cumprod()

In [ ]:
p.cum_return().head(2)

In [ ]:
#| export
@patch
def real_r(self: Portfolio, components=False):
    'Real (inflation-adjusted) return'
    if self.cpi is None:
        raise ValueError("You must specify CPI data to compute real returns")
    r = self.port_rets(excess=False)
    if components: r = self.return_streams.join(r)
    real_r = (1+r).div(1+self.cpi, axis=0)-1
    return real_r.dropna()

We only get inflation adjusted returns based on when our inflation data exists.

In [ ]:
p.availability

In [ ]:
p.real_r().head(2)

Since cpi data starts in 1955 we should expect real returns to only show up in 1955 even though we have returns data since 1953

In [ ]:
#| export
@patch
def real_w(self:Portfolio):
    'Real total wealth, your total compounded wealth less inflation'
    r = self.real_r()
    real_w = (1+r).cumprod()
    real_w.name = self.name
    return real_w

In [ ]:
p.real_w().head(2)

In [ ]:
#| export
@patch
def roll_return(self:Portfolio, months=12, excess=True, extras=False, forward=False):
    'Rolling return (linearly summed) expressed in annualised terms.'
    years = months/12
    r = self.port_rets(excess=excess)
    if extras: r = r.to_frame().join(self.rf).join(self.cpi).dropna()
    if forward: r = r[::-1]
    r = r.rolling(months).sum().dropna()
    if forward: r = r[::-1]
    return r/years

In [ ]:
p.roll_return(24).plot()

We can also ask for cpi and rf to get a better feel of returns and conditions

In [ ]:
p.roll_return(24, excess=False, extras=True).plot()

In [ ]:
#| export
@patch
def cum_excess_return(self:Portfolio):
    "Cumulative sum of monthly returns (no compounding)"
    return self.port_rets().cumsum()

In [ ]:
p.cum_excess_return().plot()

Shows how portfolio performs each month, upwards sloping means we are doing better than risk-free, downward sloping means we are doing worse. Flat means we are same. Does not use compounding.

In [ ]:
#| export
@patch
def detrended_cum_return(self:Portfolio):
    "Cumulative sum of returns minus long-run mean"
    r = self.port_rets()
    return (r - r.mean()).cumsum()

In [ ]:
p.detrended_cum_return().plot()

This makes it easier to see environmental biases: periods where an asset did worse or better than its average.

## Comparing portfolios

In [ ]:
#| export
def _align_portfolios(*portfolios):
    "Return new portfolios restricted to their common valid dates."
    ix = portfolios[0].return_streams.index
    for p in portfolios[1:]: ix = ix.intersection(p.return_streams.index)
    return (p.between(ix.min(), ix.max()) for p in portfolios)

In [ ]:
#| hide
p1 = Portfolio("60/40", rets, weights={'stocks': 0.60, 'bonds': 0.40}, rf=rf)
p2 = Portfolio("Risk Parity Short Dates", rets.loc['2000':'2010'], weights={'stocks': 0.42, 'bonds': 0.58}, rf=rf)
p1_aligned, p2_aligned = _align_portfolios(p1, p2)
assert p1_aligned.return_streams.index.equals(p2_aligned.return_streams.index)
assert p1_aligned.return_streams.index.min() == pd.Period('2000-01', freq='M')
assert p1_aligned.return_streams.index.max() == pd.Period('2010-12', freq='M')

In [ ]:
#| export
def compare(*portfolios, metric='stats', **kwargs):
    'Compares two portfolios using a metric'
    portfolios = _align_portfolios(*portfolios)
    df = pd.concat([getattr(p, metric)(**kwargs) for p in portfolios], axis=1)
    df.name = metric
    return df

In [ ]:
compare(p1,p2)

We may choose any of the above defined metrics.

In [ ]:
compare(p1,p2, metric='cum_return').plot()

We can also supply arguments to the metrics as keyword arguments.

In [ ]:
compare(p1,p2, metric='roll_return', months=24).plot()

## Portfolio inspection

Statistics on return, risk and correlation of the assets that make up a portfolio.

In [ ]:
#| export
@patch
def asset_stats(self:Portfolio):
    'Expected excess return, volatility and sharpe ratio'
    e,v,s = _sharpe(self.port_rets(agg=False))
    stats = {'expected_return': e, 'volatility': v, 'sharpe_ratio': s}
    return pd.DataFrame(stats).T

In [ ]:
p.asset_stats()

In [ ]:
#| export
@patch
def risk_contribution(self:Portfolio):
    "Per-asset risk contribution as percentage of portfolio variance"
    cov = self.port_rets(agg=False).join(self.port_rets()).cov() * 12
    pv = cov.loc[self.name, self.name]
    rc = {a: cov.loc[a, self.name] / pv for a in self.weights}
    return pd.Series(rc, name=self.name)

In [ ]:
p.risk_contribution()

If a large percentage comes from a single asset it means that this asset is dominating the volatiltiy and hence the risk-return profile of the portfolio. Each asset's contribution is dependent on its allocation weight and how much each asset correlates with the total portfolio.

In [ ]:
#| export
@patch
def correlation(self:Portfolio):
    "Pairwise correlation matrix of underlying assets and portfolio"
    return self.port_rets(agg=False).join(self.port_rets()).corr()

In [ ]:
p.correlation()

In [ ]:
#| export
@patch
def return_drivers(self:Portfolio, months=12*5):
    "Asset and portfolio rolling return"
    return self.port_rets(agg=False).join(self.port_rets()).rolling(months).agg(sum).dropna()

In [ ]:
p.return_drivers().plot()

Shows us a different picture of the risk contribution and what is driving returns. Here the equity asset is clearly driving portfolio returns.

## Drawdowns

In [ ]:
#| export
@patch
def drawdown_series(self:Portfolio, assets=False):
    "Drawdown series"
    wealth = self.cum_return()
    if assets: wealth = (1+self._port_rets(excess=False, agg=False)).cumprod().join(wealth)
    return wealth / wealth.cummax() - 1

In [ ]:
p.drawdown_series().plot()

In [ ]:
#| export
@patch
def max_drawdown(self:Portfolio):
    "Worst peak-to-trough decline"
    return self.drawdown_series().min().item()

In [ ]:
p.max_drawdown()

## Decade by decade analysis

In [ ]:
#| export
@patch
def r_by_decade(self: Portfolio):
    "Real return grouped by decade"
    decade_starts = [y for y in range(1900, 2030, 10)]
    blocks = [self.real_r().loc[f'{y}-01-01':f'{y+9}-12-31'] for y in decade_starts]
    blocks = [b for b in blocks if len(b) == 120]
    df = [pd.concat([pd.Series([1]), 1+b]).cumprod().reset_index(drop=True) for b in blocks]
    df = pd.concat(df, axis=1)
    df.columns = [f'{b.index[0].year}s' for b in blocks]
    df.index = df.index/12 # normalize months to years
    return df

In [ ]:
p.r_by_decade().plot(logy=True)

## Leverage

Creating a leveraged portfolio is easy: specify weights that add up to more than 100%

In [ ]:
p_l = Portfolio("Leveraged", rets, weights={'stocks': 0.65, 'bonds': 0.65}, rf=rf)
p_l

In [ ]:
p_u = Portfolio("Unlevered", rets, weights={'stocks': 0.50, 'bonds': 0.50}, rf=rf)
p_u

Leveraging a portfolio increases the volatility and return linearly.

In [ ]:
compare(p_l, p_u)

As a result we also get larger swings and drawdowns

In [ ]:
compare(p_u, p_l, metric='drawdown_series').plot()

In [ ]:
compare(p_u, p_l, metric='cum_return').plot(logy=True)

## Risk

In [ ]:
#| export
@patch
def rolling_vol(self:Portfolio, months=24):
    "Annualised rolling volatility per asset"
    return self.port_rets(excess=False, agg=False).rolling(months).agg(lambda x: x.std() * 12**0.5).dropna()

In [ ]:
p.rolling_vol().plot()

We can normalize these volatilities to get a simple measure of risk contribution. This ignores covariance and treats each asset as independent.

In [ ]:
#| export
@patch
def rc_simple(self:Portfolio):
    'Simple risk contribution metric'
    vol = self.rolling_vol()
    tot = vol.sum(axis=1)
    rc = vol.div(tot, axis=0)
    return rc

In [ ]:
rc = p.rc_simple()

In [ ]:
rc.plot()